<div align="center">

# 🇳🇵 Nepal GovAgent — Live Demo
### Ask Nepal's government documents anything. In Nepali or English.

[![PyPI](https://img.shields.io/pypi/v/nepal-gov-agent?color=teal)](https://pypi.org/project/nepal-gov-agent/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![GitHub](https://img.shields.io/badge/GitHub-irfanalidv%2FNepal--Gov--Agent-black?logo=github)](https://github.com/irfanalidv/Nepal-Gov-Agent)

**Built by [Irfan Ali](https://github.com/irfanalidv) · DataCortex IQ**

</div>

---

## 📁 What's available — 5 documents via `download_corpus()`

The next step downloads five Nepal government PDFs into `./nepal_gov_data/` (no separate repo checkout). Example questions you can try:

| Document | Language | Example questions |
|---|---|---|
| **National AI Policy 2082** | English | Data privacy · AI governance |
| **Constitution of Nepal** (2nd amendment) | English | Fundamental rights |
| **Digital Nepal Framework 2.0** | English | Digitization priorities |
| **प्रतिनिधि सभा निर्वाचन अध्यादेश २०८२** | Nepali 🇳🇵 | Candidate eligibility · voting |
| **मानव अधिकार पुरस्कार कोष नियमावली** | Nepali 🇳🇵 | Fund purpose · awards |

> 💡 **Add your own PDFs?** See **Use Your Own Nepal Government PDFs** near the end.

---

## How it works

Every answer includes **source citations** — document name and page. Runs offline on this machine; nothing is sent to the cloud.

> ⏱️ **First run:** Install + download takes ~2 minutes. After that, answers are instant.



> ⚠️ **Research Prototype** — This is not production-ready for government deployment. The benchmark measures retrieval quality only, not answer safety or factual correctness. See [Scope and Limitations](https://github.com/irfanalidv/Nepal-Gov-Agent#scope-and-limitations) in the README before any official use.



## 🛠️ Setup — Run once


In [1]:
# package already installed in this environment
print("✅ Installation complete")


✅ Installation complete


In [2]:
# ------------------------------------------------------------
# Option A — Download the seed corpus (same PDFs as repo Data/)
# 5 documents: AI Policy, Constitution, Digital Nepal Framework,
# election ordinance (Nepali), human rights fund rules (Nepali).
# (Seed omits the Law Commission Legal Maxims volume — see README.)
# ------------------------------------------------------------
from nepal_gov_agent import GovRAG, GovRAGConfig, download_corpus

corpus_dir = download_corpus()  # downloads to ./nepal_gov_data/
import os

# Older seed downloads included a large Legal Maxims PDF — remove if present
_legacy_maxims = os.path.join(corpus_dir, "1714977234_32.pdf")
if os.path.isfile(_legacy_maxims):
    os.remove(_legacy_maxims)
    print("Removed legacy Legal Maxims PDF from corpus folder")

# ------------------------------------------------------------
# Option B — Use your own Nepal government PDFs
# Comment out Option A above, uncomment below, point at your folder
# ------------------------------------------------------------
# corpus_dir = "./my_ministry_docs/"

_cfg = GovRAGConfig(cache_dir=os.path.join(corpus_dir, ".nepal_gov_cache"))
rag = GovRAG(corpus_dir=corpus_dir, config=_cfg)
print(f"✅ GovRAG ready — corpus: {corpus_dir}")
print("🔒 Fully offline. No data leaves this machine.")



/Users/irfanalidv/.pyenv/versions/3.12.1/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


USER_AGENT environment variable not set, consider setting it to identify your requests.


📁 Corpus folder: /Users/irfanalidv/Personal_Lib/Nepal-Gov-Agent/examples/nepal_gov_data
📥 Downloading 5 Nepal government documents...

   ✓ Already exists — skipping: 2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२_v1cs5ms.pdf
   ✓ Already exists — skipping: Constitution of Nepal (2nd amd. English)_xf33zb3.pdf
   ✓ Already exists — skipping: National AI Policy-Final_uxc94vg.pdf
   ✓ Already exists — skipping: dnf_jbji8eb.pdf
   ✓ Already exists — skipping: मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075_n4hme7v.pdf

✅ Seed corpus ready — 5 documents in /Users/irfanalidv/Personal_Lib/Nepal-Gov-Agent/examples/nepal_gov_data
💡 To use your own PDFs, pass any folder to GovRAG(corpus_dir=...)



✅ GovRAG ready — corpus: /Users/irfanalidv/Personal_Lib/Nepal-Gov-Agent/examples/nepal_gov_data
🔒 Fully offline. No data leaves this machine.


## 🔍 Live Demos — Ask Nepal's Government Documents
*Each demo below shows a real business scenario. Run any cell to see live retrieval.*

> **Optional — answer synthesis:** Install [Ollama](https://ollama.com), run `ollama pull qwen2.5:7b`, then set `use_llm=True` in `ask_and_display(...)` for concise answers with inline citations.



In [3]:
# Helper — run this once before the demo cells below
from IPython.display import Markdown, display
import re

_DOC_ALIASES = {
    "dnf_jbji8eb.pdf": "Digital Nepal Framework 2.0",
}


def _clean_doc_display(doc_id: str) -> str:
    if doc_id in _DOC_ALIASES:
        return _DOC_ALIASES[doc_id]
    base = doc_id.replace(".pdf", "").replace(".PDF", "")
    if "_" in base:
        prefix, suffix = base.rsplit("_", 1)
        if suffix.isalnum() and len(suffix) >= 5:
            return prefix
    return base


def ask_and_display(question, context="", *, rag_client=None, use_llm=False, allowed_doc_ids=None):
    """If use_llm=True, tries Ollama (qwen2.5:7b) for grounded synthesis with citations."""
    g = rag if rag_client is None else rag_client
    if context:
        display(Markdown(f"> 💼 **Scenario:** {context}"))
    display(Markdown(f"**❓ Question:** `{question}`"))

    if use_llm:
        try:
            from nepal_gov_agent import OllamaClient

            result = g.ask(
                question,
                llm=OllamaClient(),
                with_citations=True,
                allowed_doc_ids=allowed_doc_ids,
            )
        except Exception as e:
            display(Markdown(f"*LLM unavailable ({e!s}) — showing retrieval-only answer.*"))
            result = g.ask(question, allowed_doc_ids=allowed_doc_ids)
    else:
        result = g.ask(question, allowed_doc_ids=allowed_doc_ids)

    answer = result.answer or ""
    if not use_llm:
        answer = re.sub(r"\[[^\]]*?\]\s*Block\s*\d+\s*\n", "", answer)
        answer = re.sub(
            r"^www\.lawcommission\.gov\.np\s*\n", "", answer, flags=re.MULTILINE
        )
        answer = re.sub(r"^\d+\s*\n", "", answer, flags=re.MULTILINE)
        answer = re.sub(r"\n{3,}", "\n\n", answer).strip()
        if len(answer) > 500:
            truncated = answer[:500]
            last_period = truncated.rfind(".")
            if last_period > 200:
                answer = truncated[: last_period + 1] + "\n\n*[See source documents for full text]*"
            else:
                answer = truncated + "…\n\n*[See source documents for full text]*"
    else:
        answer = re.sub(r"\n{3,}", "\n\n", answer).strip()

    display(Markdown("---"))
    display(Markdown("**📋 Answer:**"))
    display(Markdown(answer))
    display(Markdown("---"))
    display(Markdown("**📌 Sources:**"))
    for src in result.sources[:3]:
        doc_clean = _clean_doc_display(src["doc"])
        display(Markdown(f"- 📄 `{doc_clean}` — Page {src['page']}"))
    display(Markdown(""))
    return result


# Optional: cleaner answers with a local LLM (fully offline on your machine):
#   brew install ollama  # or see https://ollama.com
#   ollama pull qwen2.5:7b
# Then call: ask_and_display("...", use_llm=True)



---

### Usecase 1 — National AI Policy (English)

**Scenario:** Tech entrepreneur checking data governance before launching in Nepal.

**Question:** *What does Nepal's National AI Policy say about data privacy and sovereignty?*



In [4]:
_ = ask_and_display(
    "What does Nepal's National AI Policy say about data privacy and sovereignty?",
    context="Tech entrepreneur checking data governance before launching in Nepal.",
    use_llm=True,
)


> 💼 **Scenario:** Tech entrepreneur checking data governance before launching in Nepal.

**❓ Question:** `What does Nepal's National AI Policy say about data privacy and sovereignty?`

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

---

**📌 Sources:**

- 📄 `National AI Policy-Final` — Page 15

- 📄 `National AI Policy-Final` — Page 20

- 📄 `National AI Policy-Final` — Page 25

---

### Usecase 2 — Constitutional rights (English)

**Scenario:** NGO designing a citizen services program.

**Question:** *What fundamental rights does the Constitution of Nepal guarantee?*



In [5]:
_ = ask_and_display(
    "What does the Constitution say about fundamental rights?",
    context="NGO designing a citizen services program.",
    use_llm=True,
    allowed_doc_ids={"pdf:Constitution of Nepal (2nd amd. English)_xf33zb3.pdf"},
)


> 💼 **Scenario:** NGO designing a citizen services program.

**❓ Question:** `What does the Constitution say about fundamental rights?`

---

**📋 Answer:**

[Constitution of Nepal (2nd amd. English)_xf33zb3.pdf, p.161] Block 1
www.lawcommission.gov.np 
161 
 
(8) In the event of dissolution of the House of Representatives, the powers 
exercisable by the Federal Parliament pursuant to clauses (3), (4), (6) and (7) shall be 
exercised by the National Assembly. 
(9) The President may, after the issuance of a Proclamation or order of a state 
of emergency pursuant to clause (1), issue such orders as are necessary to meet the 
exigencies. Orders so issued shall be operative with the same force and effect as law 
so long as the state of emergency is in operation. 
(10) At the time of issuance of the Proclamation or order of a state of 
emergency pursuant to clause (1), the fundamental rights as provided for in Part-3 
may be suspended until the proclamation or order is in operation. 
Provided that Article 16, sub-clauses (c) and (d) of clause (2) of Article 17, 
Article 18, clause (2) of Article 19, Articles 20, 21, 22 and 24, clause (1) of Article 
26, Articles 29, 30, 31, 32, 35, clauses (1) and (2) of Article 36, Articles 38, 39, 
clauses (2) and (3) of Article 40, Articles 41, 42, 43, 45, and the right to constitutional 
remedy in relation to such Articles pursuant to Article 46 and the right to seek the 
remedy of habeas corpus shall not be suspended. 
(11) In case any Article of this Constitution is suspended pursuant to clause 
(10), no petition may lie in any court for the enforcement of the fundamental right 
conferred by that Article nor may a question be raised in any Court in that respect. 
(12) In case, during the continuance of a Proclamation or order pursuant to this 
Article, any damage is caused to a person from any act carried out by any official in 
bad faith, the victim may, within three months from the date of termination of that 
Proclamation or order, file a petition for compensation for such damage. In case such 
petition is made, the court may order for compensation by, and punish, the perpetrator 
as provided for in the federal law. 
(13) The President may, at any time, withdraw a Proclamation or order of a 
state of emergency issuance pursuant to this Article.

---

[Constitution of Nepal (2nd amd. English)_xf33zb3.pdf, p.2] Block 2
www.lawcommission.gov.np 
2 
 
BEING COMMITTED to socialism based on democratic norms and values including the 
people's competitive multiparty democratic system of governance, civil liberties, fundamental 
rights, human rights, adult franchise, periodic elections, full freedom of the press, and 
independent, impartial and competent judiciary and concept of the rule of law in order to 
build a prosperous nation;  
 
DESIRING to fulfil aspirations for sustainable peace, good governance, development and 
prosperity through the federal democratic republican system of governance; 
 
DO HEREBY ADOPT AND PROMULGATE THIS CONSTITUTION THROUGH THE 
CONSTITUENT 
ASSEMBLY.

---

[Constitution of Nepal (2nd amd. English)_xf33zb3.pdf, p.20] Block 3
www.lawcommission.gov.np 
20 
 
health, housing, employment, food and social security for their protection, upliftment, 
empowerment and development. 
 
 
(3) The citizens with disabilities shall have the right to live with dignity and 
honour, with the identity of their diversity, and have equal access to public services 
and facilities. 
 
 
(4) Every farmer shall, in accordance with law, have the right to have access to 
land for agro activities, select and protect local seeds and agro species which have 
been used and pursued traditionally. 
 
 
(5) The families of the martyrs who have sacrificed their life, and of the 
disappeared persons and those who became disabled and injured in all people's 
movements, armed conflicts and revolutions that have been carried out for progressive 
democratic changes in Nepal, democracy fighters, conflict victims and displaced ones, 
persons with disabilities, the injured and victims shall have the right to get a 
prioritized opportunity, with justice and due respect, in education, health, 
employment, housing and social security, in accordance with law. 
43. 
Right to Social Security: The indigent citizens, incapacitated and helpless citizens, 
helpless single women, citizens with disabilities, children, citizens who cannot take care 
themselves and citizens belonging to the tribes on the verge of extinction shall have 
the right to social security, in accordance with law. 
44. 
 Rights of Consumer: (1) Every consumer shall have the right to obtain quality goods 
and services. 
(2) A person who has suffered damage from any substandard goods or services 
shall have the right to obtain compensation in accordance with law. 
45. 
Right against Exile: No citizen shall be exiled. 
46. 
Right to Constitutional Remedies: There shall be a right to obtain constitutional 
remedies in the manner set forth in Article 133 or 144 for the enforcement of the 
rights conferred by this Part. 
47. 
Implementation of Fundamental Rights: The State shall, as required, make legal 
provisions for the implementation of the rights conferred by this Part, within three 
years of the commencement of this Constitution. 
48. 
Duties of Citizens: Every citizen shall have the following duties:-

---

[Constitution of Nepal (2nd amd. English)_xf33zb3.pdf, p.3] Block 4
www.lawcommission.gov.np 
3 
 
 
Part-1 
Preliminary 
1. 
Constitution as Fundamental Law: (1) This Constitution is the fundamental law of 
Nepal. Any law inconsistent with this Constitution shall, to the extent of such 
inconsistency, be void. 
 
  
(2) It shall be the duty of every person to uphold this Constitution.  
2. 
Sovereignty and State Power: The sovereignty and State power of Nepal shall be 
vested in the Nepali people. It shall be exercised in accordance with the provisions set 
forth in this Constitution.  
3. 
Nation: All the Nepali people, with multi-ethnic, multi-lingual, multi-religious, multi-
cultural characteristics and in geographical diversities, and having common aspirations 
and being united by a bond of allegiance to national independence, territorial integrity, 
national interest and prosperity of Nepal, collectively constitute the nation. 
4. 
State of Nepal: (1) Nepal is an independent, indivisible, sovereign, secular, inclusive, 
democratic, socialism-oriented, federal democratic republican State. 
Explanation: For the purposes of this Article, "secular" means religious, 
cultural freedoms, including protection of religion and culture handed down from time 
immemorial. 
(2) The territory of Nepal shall comprise the following:- 
(a) 
The territory existing at the time of commencement of this 
Constitution, and 
(b) 
Such other territory as may be acquired after the commencement of 
this Constitution. 
5. 
National Interest: (1) Safeguarding of the freedom, sovereignty, territorial integrity, 
nationality, independence and dignity of Nepal, the rights of the Nepali people, border 
security, economic wellbeing and prosperity shall be the basic elements of the national 
interest of Nepal. 
 
  
(2) Any conduct and act contrary to the national interest shall be punishable in 
accordance with the federal law.

---

[Constitution of Nepal (2nd amd. English)_xf33zb3.pdf, p.33] Block 5
www.lawcommission.gov.np 
33 
 
(2) 
To review treaties concluded in the past, and conclude treaties, 
agreements based on equality and mutual interest. 
52. 
Obligations of the State: It shall be the obligation of the State to make Nepal a 
prosperous and affluent country by protecting and promoting fundamental rights and 
human rights, pursuing directive principles of the State and gradually implementing 
policies of the State, while keeping intact the freedom, sovereignty, territorial integrity 
and independence of Nepal. 
53. 
To Submit Report: The Government of Nepal shall submit to the President an annual 
report containing the steps taken and achievements made in the implementation of the 
directive principles, policies and obligations of the State set forth in this Part, and the 
President shall cause such report to be laid through the Prime Minister before the 
Federal Parliament. 
54. 
Provisions relating to Monitoring: There shall be a committee, in accordance with 
law, in the Federal Parliament in order to monitor and evaluate whether or not the 
directive principles, policies and obligations of the State set forth in this Part have 
been implemented progressively. 
55. 
Questions not to be raised in Court: No question shall be raised in any court as to 
whether or not any matter contained in this Part has been implemented.

---

**📌 Sources:**

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 161

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 2

- 📄 `Constitution of Nepal (2nd amd. English)` — Page 20

---

### Usecase 3 — Election ordinance (Nepali 🇳🇵)

**Scenario:** Municipality officer in Butwal checking candidate eligibility rules.

**Question:** *प्रतिनिधि सभाको निर्वाचनमा उम्मेदवार हुन के-के योग्यता चाहिन्छ?*

*(What qualifications are needed to be a candidate in the House of Representatives election?)*



In [6]:
_ = ask_and_display(
    "निर्वाचनमा उम्मेदवार हुन के चाहिन्छ?",
    context="Municipality officer in Butwal checking candidate eligibility rules.",
    use_llm=True,
    allowed_doc_ids={"pdf:2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२_v1cs5ms.pdf"},
)


> 💼 **Scenario:** Municipality officer in Butwal checking candidate eligibility rules.

**❓ Question:** `निर्वाचनमा उम्मेदवार हुन के चाहिन्छ?`

---

**📋 Answer:**

[2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२_v1cs5ms.pdf, p.1] Block 1
www.lawcommission.gov.np 
1 
 
प्रतितिति सभा सदस्य तिर्ाचि (पहिलो संशोिि) अध्यादेश, २०८२ 
 
    राजपत्रमा प्रकाशशि तमति 
  208२।09।०४ 
 
संर्ि् २०82 सालको अध्यादेश िं. 03 
प्रतितिति सभा सदस्य तिर्ाचि ऐि, २०७४ लाई संशोिि गिा बिेको अध्यादेश 
 प्रस्िार्िा  प्रतितिति सभा सदस्य तिर्ाचि ऐि, २०७४ लाई ित्काल संशोिि गिा र्ाञ्छिीय 
भएको र िाल सङ् घीय संसकोको अतिर्ेशि िभएकोले   
 
 
 
िेपालको संहर्िािको िारा ११४ को उपिारा (१) बमोशजम मशरत्रपररषकोको तसफाररसमा 
राष्ट्रपतिबाट यो अध्यादेश जारी भएको छ। 
 
1. 
संशिप् ि िाम र प्रारम्भ  (१) यस अध्यादेशको िाम "प्रतितिति सभा सदस्य तिर्ाचि 
(पहिलो संशोिि) अध्यादेश , २०८२" रिेको छ। 
 
  
(2) यो अध्यादेश िुरुरि प्रारम्भ िुिेछ। 
 
2. 
प्रतितिति सभा सदस्य तिर्ाचि ऐि, २०७४ को अिुसूचीमा संशोिि  प्रतितिति सभा सदस्य 
तिर्ाचि ऐि, २०७४ को अिुसूची-१ को सट्टा देिायको अिुसूची-१ राशिएको छ -

---

[2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२_v1cs5ms.pdf, p.2] Block 2
www.lawcommission.gov.np 
2 
 
अिुसूची-१ 
  (दफा २८ को उपदफा (५) र दफा ६० को उपदफा (६) सँग सम्बशरिि) 
उम्मेदर्ारको बरदसूचीको लातग समार्ेशी आिार 
क्र.सं. 
प्रतितितित्र् िुिे समूि 
प्रतिशि 
१. 
दतलि 
13.44 
२. 
आददर्ासी जिजाति 
28.72 
३. 
िस आया 
30.28 
४. 
मिेशी 
16.15 
५. 
थारु 
6.52 
६. 
मुशस्लम 
4.89

---

**📌 Sources:**

- 📄 `2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२` — Page 1

- 📄 `2082.9.2 प्रतिनिधि सभा सदस्य निर्वाचन (पहिलो संशोधन) अध्यादेश,२०८२` — Page 2

---

### Usecase 4 — Human Rights Award Fund (Nepali 🇳🇵)

**Scenario:** Human rights researcher understanding fund operations.

**Question:** *मानव अधिकार पुरस्कार कोषको उद्देश्य र सञ्चालन प्रक्रिया के हो?*

*(What is the purpose and process of the Human Rights Award Fund?)*

*(This demo scopes retrieval to the **Human Rights Award Fund rules** PDF only so the answer is not conflated with other Nepali gazette texts in the seed set.)*



In [7]:
import os
import shutil

# Other Nepali PDFs in the seed set can out-rank this rules booklet in hybrid search.
# Build a one-document mini-corpus so this cell demonstrates the correct source.
_hr_pdf = next(
    f
    for f in os.listdir(corpus_dir)
    if f.endswith(".pdf") and "मानव अधिकार" in f
)
_u4_dir = os.path.join(corpus_dir, ".demo_u4_corpus")
shutil.rmtree(_u4_dir, ignore_errors=True)
os.makedirs(_u4_dir, exist_ok=True)
shutil.copy2(os.path.join(corpus_dir, _hr_pdf), os.path.join(_u4_dir, _hr_pdf))

_rag_u4 = GovRAG(
    corpus_dir=_u4_dir,
    config=GovRAGConfig(cache_dir=os.path.join(_u4_dir, ".nepal_gov_cache")),
)

_ = ask_and_display(
    "मानव अधिकार पुरस्कार कोषको उद्देश्य र सञ्चालन प्रक्रिया के हो?",
    context="Human rights researcher understanding fund operations.",
    rag_client=_rag_u4,
    use_llm=True,
)


> 💼 **Scenario:** Human rights researcher understanding fund operations.

**❓ Question:** `मानव अधिकार पुरस्कार कोषको उद्देश्य र सञ्चालन प्रक्रिया के हो?`

---

**📋 Answer:**

[मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075_n4hme7v.pdf, p.1] Block 1
www.lawcommission.gov.np 
1 
 
 
मानव अधिकार पुरस्कार कोष सञ्‍चालन धनयमावली, २०७५ 
 
नेपाल‍राजपत्रमा‍प्रकाशित‍धमधत 
2075।09।09 
संिोिन 
1. 
मानव अधिकार पुरस्कार कोष सञ्‍चालन (पहिलो‍संिोिन)‍ 
धनयमावली, २०82 
 
 
 
 
‍‍‍‍‍ ‍‍‍‍2082।09।24 
 
प्रिासकीय  कायहवधि (धनयधमत गने) ऐन, २०१३ को दफा २ ले ददएको अधिकार प्रयोग गरी नेपाल 
सरकारले देिायका धनयमिरु बनाएको छ। 
1. 
संशिप्‍त नाम र प्रारम्भः ‍(१) यी धनयमिरुको नाम "मानव अधिकार पुरस्कार कोष सञ्‍चालन 
धनयमावली, २०७५" रिेको  छ।  
(२) यो धनयमावली तुरुन्त प्रारम्भ िुनेछ।  
२.  
पररभाषाः हवषय वा प्रसङ्गले अको अर्य नलागेमा यस धनयमावलीमा,-  
(क)  "अध्यि" भन्‍नाले सधमधतको अध्यि सम्झनु पछय। 
(ख) 
"उपसधमधत" भन्‍नाले धनयम ८ बमोशजम गदित पुरस्कार धसफाररस उपसधमधत 
सम्झनु पछय। 
(ग) 
"कोष" भन्‍नाले धनयम ३ बमोशजमको मानव अधिकार पुरस्कार अिय कोष 
सम्झनु पछय। 
(घ)  
"पुरस्कार" भन्‍नाले यस धनयमावली बमोशजम हवतरण गररने मानव अधिकार 
पुरस्कार सम्झनु पछय।  
(ङ) 
"मन्त्रालय" भन्‍नाले नेपाल सरकारको कानून न्याय तर्ा संसदीय माधमला 
मन्त्रालय सम्झनु पछय। 
(च)  
"सधमधत" भन्‍नाले धनयम ५ बमोशजमको मानव अधिकार पुरस्कार अिय कोष 
व्यवस्र्ापन सधमधत सम्झनु पछय।  
३.  
कोषको स्र्ापना र सञ्‍चालनः (१) मानव अधिकारको संरिण र संबर्द्यनमा उल्लेखनीय 
योगदान पुर्‍याउने व्यशि वा संस्र्ालाई प्रत्येक दुई‍वषयमा‍एक‍पटक अन्तरायश्‍िय मानव 
अधिकार ददवसको उपलक्ष्यमा नेपाल सरकारको तफयबाट मन्त्रालयले मानव अधिकार 
                                               
  
पहिलो‍संिोिनद्वारा‍संिोधित।

---

[मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075_n4hme7v.pdf, p.2] Block 2
www.lawcommission.gov.np 
2 
 
पुरस्कार प्रदान गनय मानव अधिकार पुरस्कार अिय कोष नामको एक कोष स्र्ापना 
गररएकोछ।  
(२)  
उपधनयम (१) बमोशजमको कोषमा देिाय बमोशजमका रकमिरु रिने छन्:-  
(क)  नेपाल सरकारबाट प्राप्‍त रकम,  
(ख) 
स्वदेिी व्यशि वा अन्य सङ्‍घ संस्र्ाबाट प्राप्‍त रकम,  
(ग) 
हवदेिी व्यशि, सरकार वा अन्तरायश्‍िय सङ्‍घ वा संस्र्ाबाट प्राप्‍त 
रकम,  
(घ)  
कोषको रकमबाट बढे बढाएको रकम,  
(ङ)  
अन्य स्रोतबाट प्राप्‍त रकम।  
(३)  
उपधनयम (२) को खण्ड (ग) बमोशजम हवदेिी व्यशि, सरकार वा 
अन्तरायश्‍िय सङ्‍घ वा संस्र्ाबाट कुनै रकम प्राप्‍त गनुय अशघ नेपाल सरकार, अर्य मन्त्रालयको 
स्वीकृती धलनु पनेछ।  
(४)  
उपधनयम (१) बमोशजम कोषमा प्राप्‍त िुने रकम नेपाल रा्‍ि बैङ्‍कबाट "क" 
श्रेणीको मान्यता प्राप्‍त वाशणज्य बैङ्‍क मध्येबाट सधमधतले तोकेको बैङ्‍कमा मुद्दती खाता 
खोली जम्मा गररनेछ।  
४.  
कोषको प्रयोग : (१) कोषमा जम्मा भएको रकमबाट आशजयत ब्याज यस धनयमावली बमोशजम 
प्रदान गररने पुरस्कारको प्रयोजनको लाधग प्रयोग गररनेछ।  
(२) कुनै कारणबाट उपधनयम (१) बमोशजमको कायमा कोषको रकम खचय िुन 
नसकेमा त्यस्तो रकम प्रत्येक वषयको पुष मसान्तधभत्र कोषको मूल साँवामा जोड जम्मा गरी 
राशखनेछ।  
(३) मन्त्रालयले कोषको रकमबाट आशजयत ब्याज रकम यस धनयमावली बमोशजम 
प्रदान गररने पुरस्कार बािेक अन्य कायमा प्रयोग गने छैन।  
५.  
सधमधतको गिनः (१) कोषको सञ्‍चालन तर्ा व्यवस्र्ापनको लाधग मन्त्रालयमा देिाय 
बमोशजमको एक कोष व्यवस्र्ापन सधमधत रिनेछः-  
(क)  सशचव, मन्त्रालय  
 
 
 
-‍अध्यि  
(ख)  
सिसशचव (कानून), प्रिानमन्त्री तर्ा  
 
 
मशन्त्रपररषद्को कायायलय  
 
 
-‍सदस्य  
(ग)  
सिसशचव (मानव अधिकार सम्बन्िी हवषय िेने),

---

[मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075_n4hme7v.pdf, p.3] Block 3
www.lawcommission.gov.np 
3 
 
 
 
मन्त्रालय  
 
 
 
 
-‍सदस्य  
(घ)  
सिसशचव, महिला, बालबाधलका तर्ा  
 
 
जे्‍ि नागररक मन्त्रालय  
 
 
-‍सदस्य  
(ङ)  
उपसशचव (मानव अधिकार सम्बन्िी 
 
 
 हवषय िेने), मन्त्रालय  
 
 
- सदस्य सशचव  
 (२)  सधमधतको सशचवालयको काम मन्त्रालयको मानव अधिकार सम्बन्िी काम 
िेने िाखाले गनेछ।  
(३)  
सधमधतको सशचवालयको सञ्‍चालन, पुरस्कार धसफाररस तर्ा हवतरण समारोि 
सम्बन्िी खचय मन्त्रालयको वाहषयक बजेटबाट व्यिोररनेछ।  
६.  
सधमधतको बैिकः‍(१) सधमधतको बैिक वषयको कम्तीमा दुई पटक अध्यिले तोकेको धमधत, 
समय र स्र्ानमा बस्नेछ।  
(२)  सधमधतको बैिकको अध्यिता अध्यिले गनेछ।  
(३)  सधमधतको सदस्य सशचवले सधमधतको बैिक बस्नुभन्दा कम्तीमा चौबीस घण्टा 
अगावै बैिकमा छलफल िुने हवषयसूची सहितको सूचना सबै सदस्यिरुलाई ददनु पनेछ।  
(४)  
अध्यि सहित सधमधतका तीनजना सदस्य उपशस्र्त भएमा बैिकको लाधग 
गणपूरक संख्या पुगेको माधननेछ।  
(५)  
सधमधतको बैिकमा पुरस्कार प्रदान सम्बन्िी धनणयय गनुय पदाय सवयसम्मत 
रुपमा गनुय पनेछ।  
(६)  
बैिकमा आवश्यकता अनुसार मानव अधिकार सम्बन्िी हवज्ञ वा आवश्यक 
अन्य पदाधिकारीलाई समेत आमन्त्रण गनय सहकनेछ।  
(७)  
सधमधतको सदस्य सशचवले बैिकको धनणयय अधभलेख गरी अध्यिबाट 
प्रमाशणत गराई राख्‍नु पनेछ।  
(८)  
सधमधतको बैिक सम्बन्िी अन्य कायहवधि सधमधत आफैले धनिायरण गरे 
बमोशजम िुनेछ।  
७.  
सधमधतको काम, कतयव्य र अधिकारः सधमधतको काम, कतयव्य र अधिकार देिाय बमोशजम 
िुनेछ:-  
(क)  कोषको सञ्‍चालन तर्ा व्यवस्र्ापन सम्बन्िमा आवश्यक नीधत धनिायरण गने.

---

[मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075_n4hme7v.pdf, p.4] Block 4
www.lawcommission.gov.np 
4 
 
(ख)  
कोषको रकम व्यवशस्र्त गने.  
(ग) 
पुरस्कारको स्वरुप तर्ा त्यसको रािी तोक्ने,  
(घ)  
पुरस्कार प्रदान गररने िेत्र तर्ा त्यसको आिार धनिायरण गने,  
(ङ)  
उपसधमधतले धसफाररस गरेका व्यशि वा संस्र्ािरु मध्येबाट उपयुि 
देशखएका व्यशि वा संस्र्ालाई पुरस्कार प्रदान गनय मन्त्रालय समि 
धसफाररस गने,  
(च) 
पुरस्कार प्रदान गदाय ददइने प्रमाणपत्रको ढाँचा स्वीकृत गने,  
(छ)  सधमधतले गनुय पने काम कारबािी सुचारु रुपले सञ्‍चालन गनय आवश्यकता 
अनुसार उपसधमधत गिन गने,  
(ज)  
कोष सञ्‍चालन तर्ा व्यवस्र्ापन सम्बन्िी अन्य आवश्यक कायिरू गने।  
८.  
उपसधमधतको गिनः  (१) यस धनयमावली बमोशजम पुरस्कार पाउने व्यशि वा संस्र्ाको नाम 
छनौट गरी सधमधत समि धसफाररस गनय सधमधतले अध्यि वा सधमधतको कुनै सदस्यको 
संयोजकत्वमा देिायका िेत्रिरुसाँग सम्बशन्ित हवज्ञिरू समेत रिेको बढीमा पाँच सदस्यीय 
उपसधमधत गिन गनय सक्नेछ:-  
(क)  मानव अधिकारको हवषयमा अध्ययन, अध्यापन तर्ा अनुसन्िान,  
(ख)  
कानून व्यवसाय,  
(ग)  
बालबाधलका, महिला, जे्‍ि नागररक, दधलत, अपाङ्गता भएका 
व्यशि, हपछधडएका वा जोशखममा रिेका वा धसमान्तीकृत समूिको 
मानव अधिकार संरिणको िेत्रमा गरेको योगदान,  
(घ)  
लैहङ्गक न्याय,  
(ङ)  
आम सञ्‍चार।  
(२)  उपसधमधतको काम, कतयव्य र अधिकार तर्ा काय अवधि उपसधमधत गिन 
गदायका बखत सधमधतले तोके बमोशजम िुनेछ।  
(३)  उपसधमधतको बैिक उपसधमधतको संयोजकले तोकेको धमधत, समय र 
स्र्ानमा आवश्यकता अनुसार बस्नेछ।  
(४)  उपसधमधतको बैिक सम्बन्िी अन्य कायहवधि उपसधमधत आफैले धनिायरण गरे 
बमोशजम िुनेछ।  
९.  
वस्तुधन्‍ि आिारमा नाम धसफाररस गनुय पनेः  (१) उपसधमधतले कुनै व्यशि वा संस्र्ालाई 
पुरस्कारको लाधग धसफाररस गदाय त्यस्तो व्यशि वा संस्र्ाले मानव अधिकारको संरिण र

---

[मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075_n4hme7v.pdf, p.5] Block 5
www.lawcommission.gov.np 
5 
 
संबर्द्यनको लाधग पुर्‍याएको िोस योगदान, त्यस्तो व्यशि वा संस्र्ाको काय िेत्र, हविेषज्ञता, 
नैधतकता, मानव अधिकार र मानवता प्रधतको समपयण जस्ता कुरािरु स्प्‍ट रुपमा खुलाई 
वस्तुधन्‍ि आिारमा नाम धसफाररस गनुय पनेछ।  
(२) उपधनयम (१) बमोशजम उपसधमधतले पुरस्कारको धसफाररस गदाय मानव 
अधिकारसाँग सम्बशन्ित िेत्रमध्ये कुन िेत्रमा उल्लेखनीय योगदान पुर्‍याएको िो सो कुरा 
समेत खुलाई बढीमा तीनजना व्यशि वा संस्र्ाको नाम धसफाररस गनुय पनेछ।  
१०.  
पुरस्कार धसफाररसका आिार: (१) यस धनयमावली बमोशजम कुनै व्यशि वा संस्र्ालाई 
पुरस्कारको लाधग धसफाररस गदाय त्यस्तो व्यशि वा संस्र्ाले मानव अधिकारको संरिण र 
संबर्द्यनमा पुर्‍याएको योगदानको आिारमा पुरस्कारको धसफाररस गनुय पनेछ।  
(२)  उपधनयम (१) बमोशजम पुरस्कृत िुने व्यशि वा संस्र्ाको योगदानको 
मूल्याङ्कन गदाय हवचार गनुय पने कुरािरु र त्यसको आिार सधमधतले धनिायरण गरे बमोशजम 
िुनेछ।  
 
(३)  पुरस्कारको लाधग व्यशिको नाम धसफाररस गदाय उमेर, िमय, वणय, धलङ्ग, जात, 
जाधत, लैहङ्गकता, राजनीधतक वा वैचाररक आस्र्ा जस्ता कुरािरुको आिारमा भेदभाव गनुय 
िुाँदैन।  
(४)  सधमधतले पुरस्कारको लाधग व्यशि वा संस्र्ाको नाम धसफाररस गनुय अशघ 
पुरस्कारको लाधग योग्य व्यशि वा संस्र्ाको नाम सूचनामा तोहकएको समयावधिधभत्र उपलब्ि 
गराउन कम्तीमा दुईवटा राश्‍ियस्तरका दैधनक पधत्रकामा सूचना प्रकािन गरी सावयजधनक 
रुपमा आव्िान गनुय पनेछ र सो सूचना मन्त्रालयको वेबसाइटमा समेत प्रकािन गनुय पनेछ।  
११.  
पुरस्कारको लाधग धसफाररस गनय निुने :‍ (१) यस धनयमावलीमा अन्यत्र जुनसुकै कुरा 
लेशखएको भए तापधन मानव अधिकार उल्लङ्घन सम्बन्िी फौजदारी कसूरमा सजाय पाएको, 
प्रचधलत कानून बमोशजम दामासािी वा कालो सूचीमा परेको, बाल हिंसा, घरेलु वा लैहङ्गक हिंसा 
गरेको कारणले सजाय पाएको वा िधतपूधतय धतरेको, प्रचधलत कानून बमोशजम कर चुिा 
नगरेको कारणबाट कारबािीमा परेको व्यशि वा संस्र्ालाई पुरस्कारको लाधग धसफाररस गनय 
िुाँदैन।  
(२)  उपधनयम (१) बमोशजम पुरस्कारको लाधग धसफाररस गनय निुने व्यशि वा 
संस्र्ालाई पुरस्कारको धसफाररस भई धनजले त्यस्तो पुरस्कार प्राप्‍त गरेमा मन्त्रालयले जुनसुकै 
बखत त्यस्तो पुरस्कार रकम र सो सम्बन्िी प्रमाणपत्र हफताय गनय लगाउन वा सो बराबरको

---

**📌 Sources:**

- 📄 `मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075` — Page 1

- 📄 `मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075` — Page 2

- 📄 `मानव अधिकार पुरस्कार कोष सञ्चालन नियमावली, 2075` — Page 3

---

## ✏️ Try Your Own Question

Type any question below in English or Nepali and run the cell.



In [8]:
# Change this to anything you want to ask
YOUR_QUESTION = "What role does the National AI Centre play in Nepal?"

# Examples in Nepali:
# YOUR_QUESTION = "नेपालको संविधानमा शिक्षाको अधिकार कसरी परिभाषित गरिएको छ?"
# YOUR_QUESTION = "डिजिटल नेपाल फ्रेमवर्कको उद्देश्य के हो?"

_ = ask_and_display(YOUR_QUESTION)



**❓ Question:** `What role does the National AI Centre play in Nepal?`

---

**📋 Answer:**

Related to Strategy 8.3 (Develop advanced, high-capacity infrastructures 
related to AI) 
9.12 Improve existing information and communication technology (ICT) 
infrastructure to make it AI-friendly. 
9.13 Make provision for infrastructures required for AI development and 
application, including Data Centre, Cloud Infrastructure, high-performance 
computing, reliable electricity supply and high-speed internet. 
9.

*[See source documents for full text]*

---

**📌 Sources:**

- 📄 `National AI Policy-Final` — Page 11

- 📄 `National AI Policy-Final` — Page 18

- 📄 `National AI Policy-Final` — Page 20

---

## 📊 Retrieval Validation

These are real benchmark results run against the 5-document seed corpus.
Recall@3 = 0.857 means: for 6 out of 7 test queries, the correct document
section appeared in the top 3 retrieved results.

> This measures **retrieval quality only** — not answer safety or correctness.
> See [Scope and Limitations](https://github.com/irfanalidv/Nepal-Gov-Agent#scope-and-limitations).



In [9]:
import logging

logging.getLogger("nepal_gov_agent").setLevel(logging.WARNING)

from nepal_gov_agent import run_benchmark

results = run_benchmark(rag, verbose=False)
print(results.report())



Nepal GovAgent Benchmark Results
Total queries:      7
Recall@1:           0.714
Recall@3:           1.000
Recall@5:           1.000
Keyword hit rate:   1.000
Doc hit rate:       1.000
Nepali recall@3:    1.000
English recall@3:   1.000


---

## 📂 Use Your Own Nepal Government PDFs

This library works with **any** Nepal government PDF — not just the 5 seed
documents. Ministry circulars, municipal SOPs, land records, provincial
guidelines — drop them in and ask questions.



In [10]:
import os

# Option 1 — Upload from your computer (Google Colab only)
try:
    from google.colab import files

    print("Select your Nepal government PDF(s) to upload...")
    uploaded = files.upload()

    os.makedirs("./my_ministry_docs/", exist_ok=True)
    for fname in uploaded:
        dest = os.path.join("./my_ministry_docs/", fname)
        with open(dest, "wb") as f:
            f.write(uploaded[fname])
        print(f"✅ Added: {fname}")

    print("\n🔄 Now re-run the Setup cell pointing at your folder:")
    print('   rag = GovRAG(corpus_dir="./my_ministry_docs/")')
except ImportError:
    print("Not in Google Colab — copy PDFs into a folder locally, then:")
    print('   rag = GovRAG(corpus_dir="./my_ministry_docs/")')

# Option 2 — Point at a folder you already have
# rag = GovRAG(corpus_dir="./my_ministry_docs/")



Not in Google Colab — copy PDFs into a folder locally, then:
   rag = GovRAG(corpus_dir="./my_ministry_docs/")


---

## 🤝 Contribute

The more documents in the corpus, the more useful this becomes.

**What's needed most:**
- Ministry circulars (2080–2082 BS)
- Provincial government SOPs
- Municipality service guidelines
- Land, labor, tax regulation PDFs

Open a PR: **[github.com/irfanalidv/Nepal-Gov-Agent](https://github.com/irfanalidv/Nepal-Gov-Agent)**

---

<div align="center">

**Built in the spirit of AIAN's four pillars: Data. Infrastructure. Policy. Resources.**

🇳🇵 *Working on Nepal's AI infrastructure layer? Open an issue or reach out.*

**Irfan Ali** · [LinkedIn](https://www.linkedin.com/in/irfanalidv/) ·
[GitHub](https://github.com/irfanalidv) · irfan.ali@datacortex.in

</div>

